QuACK is a pure-Python library of GPU kernels (RMSNorm, softmax, cross-entropy, GEMMs). Nothing is compiled when it is installed: each kernel is generated and compiled by the CuTe DSL the first time it is called, for the GPU it runs on. vLLM's FlashAttention-4 path and several model-specific kernels import it. Examples 1-2 run on any machine; Examples 3-5 need a GPU of the Ampere generation or newer (the CuTe DSL has no target below `sm_80`) and compare each kernel with torch. No network.

## Example 1 — the installed version, what `import quack` offers, and vLLM's own import (no GPU)

In [ ]:
import importlib.metadata
import cutlass
import quack
from quack.compile_utils import make_fake_tensor     # the import vLLM's CuTe-DSL kernels make

assert quack.__version__ == importlib.metadata.version("quack_kernels"), quack.__version__
assert {"rmsnorm", "softmax", "cross_entropy"} <= set(quack.__all__), quack.__all__
# A "fake tensor" describes a tensor's type and shape to the kernel compiler without any GPU memory behind it.
fake = make_fake_tensor(cutlass.BFloat16, (8, 1024), divisibility=8)
assert tuple(fake.shape) == (8, 1024) and fake.element_type is cutlass.BFloat16, (fake.shape, fake.element_type)
assert make_fake_tensor(None, (8,)) is None
print("quack", quack.__version__, "exports", sorted(quack.__all__), "; fake tensor", tuple(fake.shape), fake.element_type.__name__)

## Example 2 — quack's own reference RMSNorm (plain torch, no GPU) agrees with `torch.nn.RMSNorm`

In [ ]:
import torch
from quack.rmsnorm import rmsnorm_ref

torch.manual_seed(0)
x, weight = torch.randn(4, 64), torch.randn(64)
reference = rmsnorm_ref(x, weight, eps=1e-6)
torch_module = torch.nn.RMSNorm(64, eps=1e-6)
with torch.no_grad():
    torch_module.weight.copy_(weight)
difference = float((reference - torch_module(x).detach()).abs().max())
assert difference < 1e-5, difference
print("rmsnorm_ref matches torch.nn.RMSNorm on the CPU; largest difference", f"{difference:.1e}")

## Example 3 — the RMSNorm kernel, compiled on first use and run on the GPU

In [ ]:
import torch
import quack
from quack.rmsnorm import QuackRMSNorm, rmsnorm_ref

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (8, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is below the CuTe DSL's 8.0 floor -> no kernel can be compiled for this GPU")
else:
    torch.manual_seed(0)
    x = torch.randn(8, 1024, device="cuda", dtype=torch.bfloat16)
    weight = torch.randn(1024, device="cuda")
    out = quack.rmsnorm(x, weight, eps=1e-6)
    assert out.shape == x.shape and out.dtype == torch.bfloat16
    difference = float((out.float() - rmsnorm_ref(x, weight, eps=1e-6).float()).abs().max())
    assert difference < 2e-2, difference                    # bf16 keeps about 3 significant digits
    module = QuackRMSNorm(1024).cuda()                       # the drop-in torch.nn.RMSNorm replacement
    full = torch.randn(8, 1024, device="cuda")
    assert float((module(full) - torch.nn.RMSNorm(1024).cuda()(full)).abs().max()) < 1e-4
    print(f"RMSNorm kernel on sm_{capability[0]}{capability[1]}: bf16 output within {difference:.1e} of the reference; the module matches torch")

## Example 4 — the softmax and cross-entropy kernels against torch

In [ ]:
import torch
import quack

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (8, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is below the CuTe DSL's 8.0 floor -> no kernel can be compiled for this GPU")
else:
    torch.manual_seed(0)
    logits = torch.randn(16, 4096, device="cuda")
    targets = torch.randint(0, 4096, (16,), device="cuda")
    probabilities = quack.softmax(logits)
    assert float((probabilities - torch.softmax(logits, dim=-1)).abs().max()) < 1e-6
    assert float(probabilities.sum(dim=-1).sub(1).abs().max()) < 1e-5
    loss = quack.cross_entropy(logits, targets, reduction="mean")
    expected = torch.nn.functional.cross_entropy(logits, targets)
    assert abs(float(loss) - float(expected)) < 1e-4, (float(loss), float(expected))
    per_row = quack.cross_entropy(logits, targets, reduction="none")
    assert per_row.shape == (16,) and torch.allclose(per_row.mean(), expected, atol=1e-4)
    print(f"softmax rows sum to 1 and match torch; cross-entropy {float(loss):.4f} (torch {float(expected):.4f})")

## Example 5 — gradients flow through the kernels (the backward kernel is compiled too)

In [ ]:
import torch
import quack
from quack.rmsnorm import rmsnorm_ref

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
capability = torch.cuda.get_device_capability()
if capability < (8, 0):
    print(f"SKIP: compute capability {capability[0]}.{capability[1]} is below the CuTe DSL's 8.0 floor -> no kernel can be compiled for this GPU")
else:
    torch.manual_seed(0)
    x = torch.randn(8, 1024, device="cuda", requires_grad=True)
    weight = torch.randn(1024, device="cuda", requires_grad=True)
    quack.rmsnorm(x, weight).square().sum().backward()
    x_ref, weight_ref = x.detach().clone().requires_grad_(), weight.detach().clone().requires_grad_()
    rmsnorm_ref(x_ref, weight_ref).square().sum().backward()
    difference_x = float((x.grad - x_ref.grad).abs().max())
    difference_w = float((weight.grad - weight_ref.grad).abs().max())
    assert difference_x < 1e-3 and difference_w < 1e-3, (difference_x, difference_w)
    print(f"gradients match torch autograd of the reference: input {difference_x:.1e}, weight {difference_w:.1e}")